In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

# Set up the Chrome options to run in headless mode and specify a user data directory
options = Options()
options.headless = True
options.add_argument("--user-data-dir=/path/to/some/directory")  # Specify a unique path for user data

# Create a new instance of the Chrome driver
driver = webdriver.Chrome(options=options)

def get_ads_from_page(url):
    try:
        driver.get(url)
        time.sleep(5)  # Wait for the page to load
        
        # Find the number of ads
        ads_count = driver.find_element(By.CSS_SELECTOR, 'div[data-testid="results-count"]').text.split(' ')[0]
        
        # Find the start date (this may change based on the structure of the ad details)
        start_date = driver.find_element(By.CSS_SELECTOR, 'div[data-testid="ad-date"]').text
        
        data = {
            'Start Date': start_date,
            'Number of Ads': ads_count
        }
        
        return data
    except Exception as e:
        print(f"Error: {e}")
        return None

def scrape_facebook_ads(base_url: str, pages: int):
    ads_data = []

    for page_num in range(1, pages + 1):
        url = f"{base_url}&page={page_num}"
        ad_info = get_ads_from_page(url)
        
        if ad_info:
            ads_data.append(ad_info)
    
    return ads_data

# URL for a specific Facebook page on the Ads Library
base_url = 'https://www.facebook.com/ads/library/?active_status=active&ad_type=all&country=GB&is_targeted_country=false&media_type=all&search_type=page&view_all_page_id=114182943298726'

# Scrape data from 5 pages (adjust as needed)
ads = scrape_facebook_ads(base_url, pages=5)

# Convert to DataFrame
df = pd.DataFrame(ads)

# Save to CSV
df.to_csv('facebook_ads_data.csv', index=False)

# Close the driver
driver.quit()


SessionNotCreatedException: Message: session not created
from unknown error: cannot create default profile directory
Stacktrace:
#0 0x6320fa65c75a <unknown>
#1 0x6320fa10f4b0 <unknown>
#2 0x6320fa14d8cd <unknown>
#3 0x6320fa14767a <unknown>
#4 0x6320fa144ccf <unknown>
#5 0x6320fa194f0e <unknown>
#6 0x6320fa194436 <unknown>
#7 0x6320fa186363 <unknown>
#8 0x6320fa152d63 <unknown>
#9 0x6320fa1539c1 <unknown>
#10 0x6320fa621a6b <unknown>
#11 0x6320fa625951 <unknown>
#12 0x6320fa609b62 <unknown>
#13 0x6320fa6264c4 <unknown>
#14 0x6320fa5ee13f <unknown>
#15 0x6320fa64a6f8 <unknown>
#16 0x6320fa64a8d6 <unknown>
#17 0x6320fa65b5a6 <unknown>
#18 0x79d7405e0609 start_thread


In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
from datetime import datetime

def setup_driver():
    """Set up Chrome driver with options"""
    chrome_options = Options()
    # Run in headless mode for production, but visible for debugging
    # chrome_options.add_argument("--headless")  
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920x1080")
    chrome_options.add_argument("--disable-notifications")
    chrome_options.add_argument("--lang=en-GB")
    driver = webdriver.Chrome(options=chrome_options)
    return driver

def get_ads_data(driver, url):
    """Get ads data from a single page"""
    print(f"Loading page: {url}")
    driver.get(url)
    
    # Wait for the ads to load - using more reliable wait conditions
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.XPATH, "//div[contains(text(), 'Started running on') or contains(@class, 'x1lliihq')]"))
        )
    except Exception as e:
        print(f"Timed out waiting for ads to load: {e}")
        return []
    
    # Find all ad containers - using more flexible selectors
    ad_containers = driver.find_elements(By.CSS_SELECTOR, "div[role='article']") or \
                   driver.find_elements(By.CSS_SELECTOR, "div.x1lliihq") or \
                   driver.find_elements(By.CSS_SELECTOR, "div.x1yztbdb")
    
    if not ad_containers:
        print("No ad containers found")
        return []
    
    ads_data = []
    
    for ad in ad_containers:
        try:
            # Extract ad date - using multiple possible selectors
            date_element = None
            selectors_to_try = [
                "span:contains('Started running on')",
                "div[aria-label*='Started running on']",
                "div.x1lliihq",
                "span.x4k7w5x"
            ]
            
            for selector in selectors_to_try:
                try:
                    if "contains" in selector:
                        date_element = ad.find_element(By.XPATH, f".//{selector}")
                    else:
                        date_element = ad.find_element(By.CSS_SELECTOR, selector)
                    break
                except:
                    continue
            
            if not date_element:
                print("Could not find date element")
                continue
                
            date_text = date_element.text.replace("Started running on ", "").strip()
            
            # Convert date to standard format
            try:
                date_obj = datetime.strptime(date_text, "%d %b %Y")
                formatted_date = date_obj.strftime("%Y-%m-%d")
            except ValueError:
                formatted_date = date_text  # Keep original if parsing fails
            
            ads_data.append({
                'Start Date': formatted_date,
                'Ad Count': 1  # Each entry represents one ad
            })
            
        except Exception as e:
            print(f"Error processing ad: {str(e)[:200]}")  # Truncate long error messages
            continue
    
    return ads_data

def scrape_facebook_ads_library(page_id, country_code='GB', max_pages=3):
    """Scrape ads from Facebook Ads Library"""
    driver = setup_driver()
    base_url = f"https://www.facebook.com/ads/library/?active_status=active&ad_type=all&country={country_code}&is_targeted_country=false&media_type=all&search_type=page&view_all_page_id={page_id}"
    
    all_ads_data = []
    
    try:
        # First page
        print("Scraping page 1...")
        ads_data = get_ads_data(driver, base_url)
        all_ads_data.extend(ads_data)
        
        # Try to navigate through pages
        for page_num in range(2, max_pages + 1):
            print(f"Attempting to scrape page {page_num}...")
            try:
                # Try to find and click the "See more ads" or next page button
                next_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.XPATH, "//div[contains(text(), 'See more ads') or contains(text(), 'Next')]"))
                )
                driver.execute_script("arguments[0].scrollIntoView();", next_button)
                time.sleep(1)
                next_button.click()
                time.sleep(3)  # Wait for page to load
                
                # Get data from new page
                ads_data = get_ads_data(driver, driver.current_url)
                if ads_data:
                    all_ads_data.extend(ads_data)
                else:
                    print(f"No ads found on page {page_num}, stopping")
                    break
                
            except Exception as e:
                print(f"Couldn't navigate to page {page_num}: {e}")
                break
                
    finally:
        driver.quit()
    
    # Process the collected data
    if all_ads_data:
        df = pd.DataFrame(all_ads_data)
        # Group by date and count ads
        result_df = df.groupby('Start Date').size().reset_index(name='Number of Ads')
        result_df = result_df.sort_values('Start Date', ascending=False)
        return result_df
    else:
        print("No ads data was scraped.")
        return pd.DataFrame()

# Example usage
page_id = "114182943298726"  # Replace with your target page ID
ads_data = scrape_facebook_ads_library(page_id, max_pages=3)

if not ads_data.empty:
    print("\nScraped Ads Data:")
    print(ads_data)
    ads_data.to_csv('facebook_ads_data.csv', index=False)
    print("Data saved to facebook_ads_data.csv")
else:
    print("No ads data was scraped.")

SessionNotCreatedException: Message: session not created: probably user data directory is already in use, please specify a unique value for --user-data-dir argument, or don't use --user-data-dir
Stacktrace:
#0 0x5bef042bd75a <unknown>
#1 0x5bef03d704b0 <unknown>
#2 0x5bef03daa308 <unknown>
#3 0x5bef03da5ccf <unknown>
#4 0x5bef03df5f0e <unknown>
#5 0x5bef03df5436 <unknown>
#6 0x5bef03de7363 <unknown>
#7 0x5bef03db3d63 <unknown>
#8 0x5bef03db49c1 <unknown>
#9 0x5bef04282a6b <unknown>
#10 0x5bef04286951 <unknown>
#11 0x5bef0426ab62 <unknown>
#12 0x5bef042874c4 <unknown>
#13 0x5bef0424f13f <unknown>
#14 0x5bef042ab6f8 <unknown>
#15 0x5bef042ab8d6 <unknown>
#16 0x5bef042bc5a6 <unknown>
#17 0x79fc3ad1f609 start_thread
